# Lab 4 — Minimax et soumission sur ML-Arena (correction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-medecine/notebooks/pym-s8-lab-4-correction.ipynb)

*Python pour la médecine — Séance 8, projet Puissance 4.*


**Objectifs.** Évaluer une position par une heuristique ; anticiper
plusieurs coups avec l'algorithme **minimax** et l'**élagage
alpha-bêta** ; tenir la limite de **0.25 s par coup** grâce à
l'approfondissement itératif ; écrire `agent.py`, le tester comme la
plateforme, et le **soumettre** sur [ML-Arena](https://ml-arena.com/viewchallenge/198).

**Comment travailler.** Exécutez les cellules dans l'ordre. Une cellule
marquée `# À vous` contient un squelette (`...`) à compléter ; la
cellule suivante vérifie votre réponse et affiche « ✅ Exercice N
réussi ». Les cellules « Lire du code » demandent d'écrire votre
prédiction **avant** d'exécuter.

## 0. Installation

In [ ]:
import subprocess
import sys

try:
    import pettingzoo
except ImportError:  # Colab : PettingZoo n'est pas préinstallé
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pettingzoo[classic]"], check=True)

import random
import time

import matplotlib.pyplot as plt
import numpy as np
from pettingzoo.classic import connect_four_v3

random.seed(0)
print("PettingZoo prêt")

### Le code des labs précédents

La cellule suivante reprend la correction des labs 1 à 3 : exécutez-la
sans la modifier (ou remplacez-la par votre propre code).

In [ ]:
def afficher_plateau(observation):
    """Le plateau en texte : X = mes pions, O = l'adversaire, . = vide."""
    obs = np.asarray(observation)
    lignes = []
    for ligne in range(6):
        cases = []
        for col in range(7):
            if obs[ligne, col, 0] == 1:
                cases.append("X")
            elif obs[ligne, col, 1] == 1:
                cases.append("O")
            else:
                cases.append(".")
        lignes.append(" ".join(cases))
    return "\n".join(lignes)

def colonnes_libres(action_mask):
    """Les indices des colonnes autorisées (masque à 1)."""
    return [col for col, autorise in enumerate(action_mask) if autorise]

class AgentAleatoire:
    """Joue une colonne autorisée au hasard (le contrat ML-Arena)."""

    def setup(self, observation_space, action_space):
        return True

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        return random.choice(colonnes_libres(action_mask))

class AgentPondere(AgentAleatoire):
    """Au hasard, mais les colonnes du centre sont plus probables."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        colonnes = colonnes_libres(action_mask)
        poids = [self.POIDS[col] for col in colonnes]
        return random.choices(colonnes, weights=poids)[0]

def jouer_partie(agent_a, agent_b, a_commence=True, index=0):
    """Une partie appelée comme sur ML-Arena : (résultat de agent_a, nb de coups).

    Le résultat vaut +1 (agent_a gagne), 0 (nulle) ou -1 (agent_a perd).
    """
    env = connect_four_v3.env()
    env.reset()
    if a_commence:
        places = {"player_0": agent_a, "player_1": agent_b}
    else:
        places = {"player_0": agent_b, "player_1": agent_a}
    for nom, agent in places.items():
        agent.reset(nom, index)
    gains = {nom: 0.0 for nom in env.possible_agents}
    nb_coups = 0
    for nom in env.agent_iter():
        observation, reward, terminated, truncated, info = env.last()
        if terminated or truncated:
            env.step(None)  # un joueur dont la partie est finie sort ainsi
            continue
        action = places[nom].choose_action(
            observation["observation"], float(reward), False, False, info,
            action_mask=observation["action_mask"])
        env.step(action)
        nb_coups += 1
        for joueur, gain in env.rewards.items():
            gains[joueur] += gain
    env.close()
    nom_a = "player_0" if a_commence else "player_1"
    return gains[nom_a], nb_coups

def jouer_match(agent_a, agent_b, n_parties=100):
    """n parties, agent_a commence une partie sur deux. Bilan vu par agent_a."""
    env = connect_four_v3.env()
    for agent in (agent_a, agent_b):  # une fois par match, comme sur ML-Arena
        agent.setup(env.observation_space("player_0"),
                    env.action_space("player_0"))
    bilan = {"victoires": 0, "nuls": 0, "defaites": 0, "coups": []}
    for i in range(n_parties):
        resultat, nb_coups = jouer_partie(agent_a, agent_b,
                                          a_commence=(i % 2 == 0), index=i)
        if resultat > 0:
            bilan["victoires"] += 1
        elif resultat < 0:
            bilan["defaites"] += 1
        else:
            bilan["nuls"] += 1
        bilan["coups"].append(nb_coups)
    return bilan


LIGNES, COLONNES = 6, 7
VIDE, MOI, ADV = 0, 1, 2

def vers_grille(observation):
    """(6, 7, 2) -> grille 6 x 7 : 0 = vide, 1 = moi, 2 = l'adversaire."""
    obs = np.asarray(observation)
    return (obs[:, :, 0] + 2 * obs[:, :, 1]).astype(int)

def grille_depuis_texte(texte):
    """Une grille depuis 6 lignes de texte : X = moi, O = l'adversaire."""
    symboles = {".": VIDE, "X": MOI, "O": ADV}
    lignes = [l.split() for l in texte.strip().splitlines()]
    return np.array([[symboles[s] for s in l] for l in lignes], dtype=int)


def observation_depuis_grille(grille):
    """L'inverse de vers_grille : l'observation (6, 7, 2) que reçoit l'agent."""
    return np.stack([grille == MOI, grille == ADV], axis=-1).astype(np.int8)


def masque_depuis_grille(grille):
    """Le masque des colonnes non pleines, comme action_mask."""
    return (grille[0] == VIDE).astype(np.int8)

def coups_valides(grille):
    """Les colonnes dont la case du haut est vide."""
    return [col for col in range(COLONNES) if grille[0, col] == VIDE]

def ligne_libre(grille, col):
    """La ligne où tombe un pion lâché dans col, ou None si elle est pleine."""
    for ligne in range(LIGNES - 1, -1, -1):  # du bas (5) vers le haut (0)
        if grille[ligne, col] == VIDE:
            return ligne
    return None

def simuler_coup(grille, col, joueur):
    """Une COPIE de la grille où joueur a joué dans col."""
    nouvelle = grille.copy()
    nouvelle[ligne_libre(grille, col), col] = joueur
    return nouvelle

def construire_fenetres():
    """Toutes les suites de 4 cases alignées sur la grille 6 x 7."""
    fenetres = []
    for ligne in range(LIGNES):
        for col in range(COLONNES):
            for dl, dc in [(0, 1), (1, 0), (1, 1), (-1, 1)]:
                cases = [(ligne + k * dl, col + k * dc) for k in range(4)]
                if all(0 <= l < LIGNES and 0 <= c < COLONNES for l, c in cases):
                    fenetres.append(cases)
    return fenetres


FENETRES = construire_fenetres()

def est_gagnant(grille, joueur):
    """Vrai si joueur a quatre pions alignés quelque part."""
    for fenetre in FENETRES:
        if all(grille[l, c] == joueur for l, c in fenetre):
            return True
    return False

def coup_gagnant(grille, joueur):
    """Une colonne qui fait gagner joueur tout de suite, sinon None."""
    for col in coups_valides(grille):
        if est_gagnant(simuler_coup(grille, col, joueur), joueur):
            return col
    return None

def coups_surs(grille, coups):
    """Les coups après lesquels l'adversaire ne peut pas gagner tout de suite."""
    return [col for col in coups
            if coup_gagnant(simuler_coup(grille, col, MOI), ADV) is None]

class AgentRegles(AgentAleatoire):
    """Gagner, sinon bloquer, sinon un coup sûr de préférence au centre."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        grille = vers_grille(observation)
        coups = colonnes_libres(action_mask)
        for joueur in (MOI, ADV):  # 1. gagner, 2. bloquer
            col = coup_gagnant(grille, joueur)
            if col is not None:
                return col
        candidats = coups_surs(grille, coups) or coups  # 3. ne pas offrir
        poids = [self.POIDS[col] for col in candidats]  # 4. le centre
        return random.choices(candidats, weights=poids)[0]


import itertools

import pandas as pd

class AgentChronometre:
    """Enveloppe un agent et mesure la durée de chaque choose_action."""

    def __init__(self, agent):
        self.agent = agent
        self.durees = []

    def setup(self, observation_space, action_space):
        return self.agent.setup(observation_space, action_space)

    def reset(self, env_player_name, episode_index):
        return self.agent.reset(env_player_name, episode_index)

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        debut = time.perf_counter()
        action = self.agent.choose_action(observation, reward, terminated,
                                          truncated, info, action_mask)
        self.durees.append(time.perf_counter() - debut)
        return action

def tournoi(agents, n_parties=50):
    """Chaque paire d'agents joue un match ; taux de victoire ligne / colonne."""
    noms = list(agents)
    taux = pd.DataFrame(np.nan, index=noms, columns=noms)
    for a, b in itertools.combinations(noms, 2):
        bilan = jouer_match(agents[a], agents[b], n_parties)
        taux.loc[a, b] = bilan["victoires"] / n_parties
        taux.loc[b, a] = bilan["defaites"] / n_parties
    return taux

---

## 1. Anticiper : l'arbre de jeu

![Un arbre minimax de profondeur 2](https://raw.githubusercontent.com/racousin/data_science_practice/main/courseware/content/python-medecine/assets/pym-s8-projet-puissance-4/arbre-minimax.png)

Depuis la position actuelle, chacun de mes coups mène à une position,
chacune des réponses adverses à une autre, etc. : c'est l'**arbre de
jeu**. On le développe jusqu'à une **profondeur** fixée, on note les
feuilles avec une **fonction d'évaluation**, puis on remonte : à mon
tour je prends le **max** (je choisis le meilleur pour moi), au tour
adverse le **min** (l'adversaire choisit le pire pour moi).

### Lire du code 1

L'arbre ci-dessus, en listes imbriquées. Que vont afficher les deux
`print` ?

In [ ]:
def minimax_arbre(noeud, maximise):
    if isinstance(noeud, int):          # une feuille : sa valeur
        return noeud
    valeurs = [minimax_arbre(enfant, not maximise) for enfant in noeud]
    return max(valeurs) if maximise else min(valeurs)

arbre = [[3, 5], [2, 9], [0, 1]]
print(minimax_arbre(arbre, True))
print(minimax_arbre(arbre, False))

**Réponse.** `3`, puis `1`. Avec MAX à la racine : les nœuds MIN
valent 3, 2 et 0, et MAX prend 3 — la colonne 0, même si le 9 est sous
la colonne 1 (l'adversaire ne le laissera jamais arriver). Avec MIN à
la racine : les nœuds MAX valent 5, 9 et 1, MIN prend 1.

---

## 2. La fonction d'évaluation

Au bout de l'arbre, il faut noter une position qui n'est pas finie. On
reprend les 69 fenêtres : une fenêtre où je suis seul avec 3 pions est
une menace, avec 2 pions un début ; une fenêtre mixte ne vaut plus
rien.

### Exercice 1 — `score_fenetre`

`valeurs` est la liste des 4 valeurs d'une fenêtre. Renvoyez : 0 si
elle contient mes pions **et** ceux de l'adversaire ; sinon 5 pour
3 pions à moi, 2 pour 2 pions à moi, −4 pour 3 pions adverses,
−2 pour 2 pions adverses, 0 dans les autres cas. (Les fenêtres
complètes sont traitées à part, comme victoires.)

In [ ]:
def score_fenetre(valeurs):
    """Le score d'une fenêtre de 4 cases, de mon point de vue."""
    moi, adv = valeurs.count(MOI), valeurs.count(ADV)
    if moi and adv:      # fenêtre mixte : plus personne ne peut y aligner 4
        return 0
    if moi == 3:
        return 5
    if moi == 2:
        return 2
    if adv == 3:
        return -4
    if adv == 2:
        return -2
    return 0

In [ ]:
assert score_fenetre([MOI, MOI, MOI, VIDE]) == 5
assert score_fenetre([VIDE, MOI, VIDE, MOI]) == 2
assert score_fenetre([ADV, ADV, VIDE, ADV]) == -4
assert score_fenetre([ADV, VIDE, ADV, VIDE]) == -2
assert score_fenetre([MOI, MOI, MOI, ADV]) == 0
assert score_fenetre([MOI, VIDE, VIDE, VIDE]) == 0
print("✅ Exercice 1 réussi")

### Exercice 2 — `evaluer`

Additionnez `score_fenetre` sur les 69 fenêtres (les valeurs d'une
fenêtre : `[grille[l, c] for l, c in fenetre]`), et ajoutez
3 points par pion à moi dans la colonne centrale (colonne 3), −3 par
pion adverse.

In [ ]:
def evaluer(grille):
    """Somme des scores des 69 fenêtres + bonus pour la colonne centrale."""
    score = 0
    for fenetre in FENETRES:
        score += score_fenetre([grille[l, c] for l, c in fenetre])
    centre = list(grille[:, COLONNES // 2])
    return score + 3 * centre.count(MOI) - 3 * centre.count(ADV)

In [ ]:
vide = np.zeros((6, 7), dtype=int)
assert evaluer(vide) == 0
assert evaluer(simuler_coup(vide, 3, MOI)) > evaluer(simuler_coup(vide, 0, MOI)), \
    "le centre vaut plus que le bord"
trois = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
X X X . . O O
''')
assert evaluer(trois) > 0
print("évaluation :", evaluer(trois))
print("✅ Exercice 2 réussi")

---

## 3. Minimax avec élagage alpha-bêta

`alpha` est la meilleure valeur que MAX est déjà sûr d'obtenir ailleurs,
`beta` la meilleure que MIN est sûr d'obtenir. Dès que `alpha >= beta`,
les coups restants de ce nœud ne peuvent plus changer la décision : on
les **élague** (`break`). Le résultat est le même que minimax, en
visitant beaucoup moins de positions — surtout si l'on essaie les
bons coups d'abord (le centre).

Le paramètre `fin` sert à la section 5 : si l'heure `fin` est dépassée,
on lève `TempsEcoule` pour abandonner la recherche.

### Exercice 3 — `minimax`

Complétez, dans cet ordre : 1) les fins de partie (j'ai gagné :
`GAGNE + profondeur` ; l'adversaire a gagné : `-GAGNE - profondeur` ;
grille pleine : 0) ; 2) `profondeur == 0` : `evaluer(grille)` ;
3) la récursion — si `maximise`, je joue (`MOI`) et je prends le max,
sinon l'adversaire joue (`ADV`) et prend le min, en mettant à jour
`alpha` / `beta` et en élaguant.

In [ ]:
GAGNE = 1_000_000


class TempsEcoule(Exception):
    """Levée quand la recherche dépasse son budget de temps."""


def minimax(grille, profondeur, alpha, beta, maximise, fin=float("inf")):
    """La valeur de la position, MOI cherchant le max et ADV le min."""
    if time.perf_counter() > fin:
        raise TempsEcoule
    if est_gagnant(grille, MOI):
        return GAGNE + profondeur  # gagner tôt vaut mieux que gagner tard
    if est_gagnant(grille, ADV):
        return -GAGNE - profondeur
    coups = coups_valides(grille)
    if not coups:
        return 0                   # grille pleine : nulle
    if profondeur == 0:
        return evaluer(grille)
    coups.sort(key=lambda col: abs(col - 3))  # le centre d'abord : plus d'élagage
    if maximise:
        valeur = -float("inf")
        for col in coups:
            valeur = max(valeur, minimax(simuler_coup(grille, col, MOI),
                                         profondeur - 1, alpha, beta, False, fin))
            alpha = max(alpha, valeur)
            if alpha >= beta:
                break              # ADV ne laissera jamais arriver ici
        return valeur
    valeur = float("inf")
    for col in coups:
        valeur = min(valeur, minimax(simuler_coup(grille, col, ADV),
                                     profondeur - 1, alpha, beta, True, fin))
        beta = min(beta, valeur)
        if alpha >= beta:
            break                  # MOI ne laissera jamais arriver ici
    return valeur

In [ ]:
inf = float("inf")
assert minimax(trois, 0, -inf, inf, True) == evaluer(trois)
gagne = simuler_coup(trois, 3, MOI)
assert minimax(gagne, 3, -inf, inf, False) == GAGNE + 3
# au tour de MOI, profondeur 1 : il voit sa victoire en colonne 3
assert minimax(trois, 1, -inf, inf, True) == GAGNE
# au tour d'ADV, profondeur 2 : il bloque, plus de victoire immédiate
assert minimax(trois, 2, -inf, inf, False) < GAGNE
print("✅ Exercice 3 réussi")

### Exercice 4 — `meilleur_coup`

À la racine, on veut le **coup**, pas seulement la valeur. Pour chaque
colonne de `coups` (dans l'ordre donné), calculez la valeur de la
position après mon coup (`minimax` avec `maximise=False`, profondeur
− 1, en passant le meilleur `alpha` trouvé jusqu'ici) ; gardez la
meilleure. Renvoyez `(colonne, valeur)`.

In [ ]:
def meilleur_coup(grille, coups, profondeur, fin=float("inf")):
    """(colonne, valeur) du meilleur coup, en essayant les coups dans l'ordre."""
    meilleur, alpha = coups[0], -float("inf")
    for col in coups:
        valeur = minimax(simuler_coup(grille, col, MOI), profondeur - 1,
                         alpha, float("inf"), False, fin)
        if valeur > alpha:
            meilleur, alpha = col, valeur
    return meilleur, alpha

In [ ]:
bloquer = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . X .
O O O . X X .
''')
assert meilleur_coup(bloquer, coups_valides(bloquer), 2)[0] == 3, "bloquer en 3"
ouverte = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O O . . . .
. X X . . . .
''')
col, valeur = meilleur_coup(ouverte, coups_valides(ouverte), 3)
assert col == 3 and valeur >= GAGNE, ". X X X . : victoire forcée en 3 coups"
print("✅ Exercice 4 réussi")

### Trouvez l'erreur

Ce minimax (sans élagage, pour faire court) ne voit aucun danger sur
la grille `bloquer` : comparez ses valeurs à celles de votre `minimax`.
Pourquoi les coups 0, 1, 2, 4, 5 et 6, qui laissent les O gagner en
colonne 3, ne valent-ils pas −1 000 000 ? Trouvez la ligne fautive.

In [ ]:
def minimax_bug(grille, profondeur, maximise):
    if est_gagnant(grille, MOI):
        return GAGNE
    if est_gagnant(grille, ADV):
        return -GAGNE
    if profondeur == 0 or not coups_valides(grille):
        return evaluer(grille)
    valeurs = [minimax_bug(simuler_coup(grille, col, MOI), profondeur - 1,
                           not maximise)
               for col in coups_valides(grille)]
    return max(valeurs) if maximise else min(valeurs)

inf = float("inf")
for col in coups_valides(bloquer):
    apres = simuler_coup(bloquer, col, MOI)
    print(f"colonne {col} : bug {minimax_bug(apres, 1, False):9d}   "
          f"correct {minimax(apres, 1, -inf, inf, False):9d}")

**Réponse.** `simuler_coup(grille, col, MOI)` pose **toujours mon**
pion, même au tour de l'adversaire : dans cet arbre, l'adversaire ne
joue jamais, donc ne gagne jamais, et aucun coup ne paraît dangereux.
Il faut `MOI if maximise else ADV`.

### Combien de positions l'élagage évite-t-il ?

On compte les appels à `simuler_coup` (une position visitée chacun)
pour minimax sans élagage (`alpha`/`beta` jamais utilisés) et avec.

In [ ]:
def minimax_naif(grille, profondeur, maximise):
    """Minimax sans élagage."""
    if est_gagnant(grille, MOI):
        return GAGNE + profondeur
    if est_gagnant(grille, ADV):
        return -GAGNE - profondeur
    coups = coups_valides(grille)
    if not coups:
        return 0
    if profondeur == 0:
        return evaluer(grille)
    joueur = MOI if maximise else ADV
    valeurs = [minimax_naif(simuler_coup(grille, col, joueur), profondeur - 1,
                            not maximise) for col in coups]
    return max(valeurs) if maximise else min(valeurs)


def compter_positions(fonction, *args):
    """Le nombre d'appels à simuler_coup pendant fonction(*args)."""
    global simuler_coup
    original, compteur = simuler_coup, [0]

    def compte(*a):
        compteur[0] += 1
        return original(*a)

    simuler_coup = compte
    try:
        fonction(*args)
    finally:
        simuler_coup = original
    return compteur[0]


inf = float("inf")
depart = simuler_coup(np.zeros((6, 7), dtype=int), 3, ADV)
for p in (2, 3, 4):
    naif = compter_positions(minimax_naif, depart, p, True)
    elague = compter_positions(minimax, depart, p, -inf, inf, True)
    print(f"profondeur {p} : {naif:6d} positions sans élagage, {elague:5d} avec")

Sans élagage, il y a environ $b^d$ positions pour un facteur de
branchement $b$ (7 colonnes) et une profondeur $d$ : chaque niveau
multiplie le travail par 7. Avec un bon ordre des coups, l'élagage
ramène ce nombre vers $b^{d/2}$ : à temps égal, on va presque deux
fois plus profond.

### Exercice 5 — `AgentMinimax`

Écrivez l'agent : `__init__(self, profondeur=4)` retient la profondeur
(sans appeler `super().__init__`, `AgentAleatoire` n'en a pas) ;
`choose_action` trie les coups autorisés « centre d'abord » — avec un
tirage au hasard entre colonnes à même distance du centre, pour ne pas
jouer toujours la même partie : `coups.sort(key=lambda col:
(abs(col - 3), random.random()))` — puis renvoie la colonne de
`meilleur_coup`.

In [ ]:
class AgentMinimax(AgentAleatoire):
    """Minimax alpha-bêta à profondeur fixe."""

    def __init__(self, profondeur=4):
        self.profondeur = profondeur

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        grille = vers_grille(observation)
        coups = colonnes_libres(action_mask)
        # centre d'abord, hasard entre colonnes à même distance du centre
        coups.sort(key=lambda col: (abs(col - 3), random.random()))
        return meilleur_coup(grille, coups, self.profondeur)[0]

In [ ]:
agent = AgentMinimax(3)
obs, masque = observation_depuis_grille(bloquer), masque_depuis_grille(bloquer)
assert agent.choose_action(obs, action_mask=masque) == 3
assert agent.choose_action(obs, terminated=True, action_mask=masque) is None
bilan = jouer_match(AgentMinimax(4), AgentRegles(), 10)
print("minimax(4) contre règles :", bilan["victoires"], "victoires,",
      bilan["defaites"], "défaites sur 10")
assert bilan["victoires"] >= 7
print("✅ Exercice 5 réussi")

---

## 4. Profondeur contre temps

Plus profond = plus fort, mais plus lent. Mesurons le temps par coup
pour chaque profondeur, sur des parties contre `AgentRegles`.
La limite de la plateforme est **0.25 s** par appel,
aller-retour compris, sur une machine qui peut être plus lente que
celle-ci : visez **moins de 0,1 s** de calcul.

In [ ]:
resultats = []
for p in range(1, 6):
    chrono = AgentChronometre(AgentMinimax(p))
    bilan = jouer_match(chrono, AgentRegles(), 4)
    d = np.array(chrono.durees)
    resultats.append((p, d.mean(), d.max(), bilan["victoires"]))
    print(f"profondeur {p} : moyenne {d.mean() * 1000:7.1f} ms, "
          f"max {d.max() * 1000:7.1f} ms, {bilan['victoires']}/4 victoires")

p, moy, mx, _ = zip(*resultats)
plt.semilogy(p, mx, "o-", label="max", color="#c0392b")
plt.semilogy(p, moy, "o-", label="moyenne", color="#2f5d9e")
plt.axhline(0.25, ls="--", color="gray", label="limite ML-Arena (0,25 s)")
plt.xlabel("profondeur")
plt.ylabel("secondes par coup (échelle log)")
plt.legend()
plt.show()

**Lecture.** Le temps est multiplié par un facteur constant à chaque
niveau (une droite en échelle logarithmique) : c'est le $b^d$. Le
**maximum** compte plus que la moyenne, car un seul appel trop long
fait perdre le match ; il survient en milieu de partie, quand
beaucoup de colonnes sont ouvertes. Une profondeur fixe est donc
risquée : trop prudente en début et en fin de partie, trop lente au
milieu.

### Où passe le temps ?

Le profileur `cProfile` compte le temps passé dans chaque fonction.

In [ ]:
import cProfile
import pstats

milieu = grille_depuis_texte('''
. . . . . . .
. . . . . . .
. . . X . . .
. . O O . . .
. X X O . . .
O X O X . . .
''')
cProfile.run("meilleur_coup(milieu, coups_valides(milieu), 4)", "profil.out")
pstats.Stats("profil.out").sort_stats("tottime").print_stats(6)

**Lecture.** L'essentiel du temps est dans `est_gagnant` et
`evaluer`, qui parcourent les 69 fenêtres à chaque position. Pistes
d'optimisation : ne vérifier la victoire que sur les fenêtres qui
passent par le dernier pion posé ; mettre en cache les positions déjà
évaluées ; représenter la grille par deux entiers (*bitboards*), où
une victoire se teste en quelques opérations binaires.

---

## 5. Tenir le temps : l'approfondissement itératif

Au lieu de fixer la profondeur, on cherche à profondeur 1, puis 2, puis
3… tant qu'il reste du temps. Quand le budget est épuisé, `minimax`
lève `TempsEcoule` : on abandonne la profondeur en cours et on joue le
meilleur coup de la **dernière profondeur terminée**. Bonus : on essaie
d'abord le meilleur coup de la profondeur précédente, ce qui améliore
l'élagage.

### Lire du code 2

Le mécanisme, sur un exemple jouet où le « temps » est remplacé par un
compteur. Que va afficher ce code ?

In [ ]:
def recherche(budget):
    meilleur = None
    for profondeur in range(1, 10):
        try:
            if profondeur * profondeur > budget:   # coût de cette profondeur
                raise TempsEcoule
            meilleur = profondeur
        except TempsEcoule:
            break
    return meilleur

print(recherche(10), recherche(50), recherche(0))

**Réponse.** `3 7 None`. Les profondeurs 1, 2, 3 « coûtent » 1, 4, 9
≤ 10 ; la 4 coûterait 16 : l'exception interrompt la boucle et l'on
garde 3, la dernière profondeur **terminée**. Avec un budget nul, rien
n'est terminé : d'où, dans `choisir_coup`, une valeur de départ
(`choix = ordre[0]`) qui est toujours un coup autorisé.

### Exercice 6 — `choisir_coup`

Complétez la boucle : pour chaque profondeur, appelez
`meilleur_coup(grille, ordre, profondeur, fin)` dans un `try` ; sur
`TempsEcoule`, sortez de la boucle ; sinon retenez le coup, placez-le
en tête de `ordre`, et arrêtez si la valeur est une victoire ou une
défaite forcée (`abs(valeur) >= GAGNE`).

In [ ]:
def choisir_coup(grille, coups, temps_max):
    """Approfondissement itératif : profondeur 1, 2, 3... tant qu'il reste du temps."""
    fin = time.perf_counter() + temps_max
    ordre = sorted(coups, key=lambda col: (abs(col - 3), random.random()))
    choix = ordre[0]
    cases_vides = int((grille == VIDE).sum())
    for profondeur in range(1, cases_vides + 1):
        try:
            choix, valeur = meilleur_coup(grille, ordre, profondeur, fin)
        except TempsEcoule:
            break          # on garde le coup de la dernière profondeur finie
        ordre = [choix] + [col for col in ordre if col != choix]
        if abs(valeur) >= GAGNE:
            break          # issue forcée trouvée : inutile de chercher plus loin
    return choix

In [ ]:
for grille in (np.zeros((6, 7), dtype=int), milieu, bloquer):
    debut = time.perf_counter()
    col = choisir_coup(grille, coups_valides(grille), 0.08)
    duree = time.perf_counter() - debut
    assert col in coups_valides(grille)
    assert duree < 0.15, f"{duree:.3f} s : le budget n'est pas respecté"
assert choisir_coup(bloquer, coups_valides(bloquer), 0.08) == 3
print("✅ Exercice 6 réussi")

---

## 6. Le fichier `agent.py`

La plateforme ne reçoit qu'un fichier : `agent.py`, qui doit contenir
**tout** ce dont l'agent a besoin, et n'importer que la bibliothèque
standard et NumPy (PettingZoo n'est **pas** installé côté agent). La
cellule suivante l'écrit avec `%%writefile` ; `TEMPS_RECHERCHE` vaut
0,08 s, une marge confortable sous les 0.25 s.

In [ ]:
%%writefile agent.py
"""Agent Puissance 4 pour ML-Arena (challenge 198).

Minimax alpha-bêta, approfondissement itératif sous budget de temps.
Bibliothèque standard + NumPy uniquement : pas de PettingZoo sur la plateforme.
"""
import random
import time

import numpy as np

TEMPS_RECHERCHE = 0.08  # s de calcul par coup ; la plateforme coupe à 0.25 s


LIGNES, COLONNES = 6, 7
VIDE, MOI, ADV = 0, 1, 2

def vers_grille(observation):
    """(6, 7, 2) -> grille 6 x 7 : 0 = vide, 1 = moi, 2 = l'adversaire."""
    obs = np.asarray(observation)
    return (obs[:, :, 0] + 2 * obs[:, :, 1]).astype(int)

def coups_valides(grille):
    """Les colonnes dont la case du haut est vide."""
    return [col for col in range(COLONNES) if grille[0, col] == VIDE]

def ligne_libre(grille, col):
    """La ligne où tombe un pion lâché dans col, ou None si elle est pleine."""
    for ligne in range(LIGNES - 1, -1, -1):  # du bas (5) vers le haut (0)
        if grille[ligne, col] == VIDE:
            return ligne
    return None

def simuler_coup(grille, col, joueur):
    """Une COPIE de la grille où joueur a joué dans col."""
    nouvelle = grille.copy()
    nouvelle[ligne_libre(grille, col), col] = joueur
    return nouvelle

def construire_fenetres():
    """Toutes les suites de 4 cases alignées sur la grille 6 x 7."""
    fenetres = []
    for ligne in range(LIGNES):
        for col in range(COLONNES):
            for dl, dc in [(0, 1), (1, 0), (1, 1), (-1, 1)]:
                cases = [(ligne + k * dl, col + k * dc) for k in range(4)]
                if all(0 <= l < LIGNES and 0 <= c < COLONNES for l, c in cases):
                    fenetres.append(cases)
    return fenetres


FENETRES = construire_fenetres()

def est_gagnant(grille, joueur):
    """Vrai si joueur a quatre pions alignés quelque part."""
    for fenetre in FENETRES:
        if all(grille[l, c] == joueur for l, c in fenetre):
            return True
    return False


def score_fenetre(valeurs):
    """Le score d'une fenêtre de 4 cases, de mon point de vue."""
    moi, adv = valeurs.count(MOI), valeurs.count(ADV)
    if moi and adv:      # fenêtre mixte : plus personne ne peut y aligner 4
        return 0
    if moi == 3:
        return 5
    if moi == 2:
        return 2
    if adv == 3:
        return -4
    if adv == 2:
        return -2
    return 0

def evaluer(grille):
    """Somme des scores des 69 fenêtres + bonus pour la colonne centrale."""
    score = 0
    for fenetre in FENETRES:
        score += score_fenetre([grille[l, c] for l, c in fenetre])
    centre = list(grille[:, COLONNES // 2])
    return score + 3 * centre.count(MOI) - 3 * centre.count(ADV)

GAGNE = 1_000_000


class TempsEcoule(Exception):
    """Levée quand la recherche dépasse son budget de temps."""


def minimax(grille, profondeur, alpha, beta, maximise, fin=float("inf")):
    """La valeur de la position, MOI cherchant le max et ADV le min."""
    if time.perf_counter() > fin:
        raise TempsEcoule
    if est_gagnant(grille, MOI):
        return GAGNE + profondeur  # gagner tôt vaut mieux que gagner tard
    if est_gagnant(grille, ADV):
        return -GAGNE - profondeur
    coups = coups_valides(grille)
    if not coups:
        return 0                   # grille pleine : nulle
    if profondeur == 0:
        return evaluer(grille)
    coups.sort(key=lambda col: abs(col - 3))  # le centre d'abord : plus d'élagage
    if maximise:
        valeur = -float("inf")
        for col in coups:
            valeur = max(valeur, minimax(simuler_coup(grille, col, MOI),
                                         profondeur - 1, alpha, beta, False, fin))
            alpha = max(alpha, valeur)
            if alpha >= beta:
                break              # ADV ne laissera jamais arriver ici
        return valeur
    valeur = float("inf")
    for col in coups:
        valeur = min(valeur, minimax(simuler_coup(grille, col, ADV),
                                     profondeur - 1, alpha, beta, True, fin))
        beta = min(beta, valeur)
        if alpha >= beta:
            break                  # MOI ne laissera jamais arriver ici
    return valeur

def meilleur_coup(grille, coups, profondeur, fin=float("inf")):
    """(colonne, valeur) du meilleur coup, en essayant les coups dans l'ordre."""
    meilleur, alpha = coups[0], -float("inf")
    for col in coups:
        valeur = minimax(simuler_coup(grille, col, MOI), profondeur - 1,
                         alpha, float("inf"), False, fin)
        if valeur > alpha:
            meilleur, alpha = col, valeur
    return meilleur, alpha

def choisir_coup(grille, coups, temps_max):
    """Approfondissement itératif : profondeur 1, 2, 3... tant qu'il reste du temps."""
    fin = time.perf_counter() + temps_max
    ordre = sorted(coups, key=lambda col: (abs(col - 3), random.random()))
    choix = ordre[0]
    cases_vides = int((grille == VIDE).sum())
    for profondeur in range(1, cases_vides + 1):
        try:
            choix, valeur = meilleur_coup(grille, ordre, profondeur, fin)
        except TempsEcoule:
            break          # on garde le coup de la dernière profondeur finie
        ordre = [choix] + [col for col in ordre if col != choix]
        if abs(valeur) >= GAGNE:
            break          # issue forcée trouvée : inutile de chercher plus loin
    return choix


class Agent:
    """Le contrat ML-Arena : setup une fois, reset à chaque partie, choose_action."""

    def setup(self, observation_space, action_space):
        return True

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        grille = vers_grille(observation)
        masque = np.asarray(action_mask)
        coups = [col for col in range(COLONNES) if masque[col]]
        return int(choisir_coup(grille, coups, TEMPS_RECHERCHE))

### Exercice 7 — Tester `agent.py` comme la plateforme

La cellule suivante importe le fichier, puis le fait jouer avec
`jouer_match` (les appels de ML-Arena) : 100 parties contre
l'aléatoire, 20 contre l'agent à règles, en mesurant **chaque** coup.
Elle vérifie aussi qu'une observation sous forme de listes est
acceptée. Votre `agent.py` doit gagner au moins 95 % contre
l'aléatoire, battre l'agent à règles le plus souvent, et ne jamais
dépasser 0,2 s.

In [ ]:
import importlib

import agent as mon_agent

importlib.reload(mon_agent)
assert "pettingzoo" not in open("agent.py", encoding="utf-8").read()

a = mon_agent.Agent()
assert a.setup({}, {}) is True and a.reset("player_1", 0) is True
vide = np.zeros((6, 7, 2), dtype=np.int8)
assert a.choose_action(vide.tolist(), 0.0, False, False, {}, [1] * 7) in range(7)

chrono = AgentChronometre(mon_agent.Agent())
contre_aleatoire = jouer_match(chrono, AgentAleatoire(), 100)
contre_regles = jouer_match(chrono, AgentRegles(), 20)
d = np.array(chrono.durees)
print("contre l'aléatoire :", {k: v for k, v in contre_aleatoire.items() if k != "coups"})
print("contre les règles  :", {k: v for k, v in contre_regles.items() if k != "coups"})
print(f"temps par coup : moyenne {d.mean() * 1000:.0f} ms, max {d.max() * 1000:.0f} ms")
assert contre_aleatoire["victoires"] >= 95
assert contre_regles["victoires"] > contre_regles["defaites"]
assert d.max() < 0.2, "trop lent : baissez TEMPS_RECHERCHE"
print("✅ Exercice 7 réussi")

---

## 7. Soumettre sur ML-Arena

Le challenge : [https://ml-arena.com/viewchallenge/198](https://ml-arena.com/viewchallenge/198). Sa page décrit le
contrat, les contraintes et le classement ELO.

Il faut une clé d'API : sur ML-Arena, **Profil → API Keys** (elle
commence par `mlk_user_`). **Ne collez jamais la clé dans une
cellule** : un notebook se partage avec son code et ses sorties. Dans
Colab, ajoutez-la dans le panneau **Secrets** (l'icône de clé à
gauche) sous le nom `MLARENA_API_KEY` et autorisez ce notebook à y
accéder. Hors de Colab, définissez la variable d'environnement
`MLARENA_API_KEY` avant de lancer Jupyter.

In [ ]:
import os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlarena-sdk==3.0.0"], check=True)
    from google.colab import userdata
    os.environ["MLARENA_API_KEY"] = userdata.get("MLARENA_API_KEY")

CHALLENGE_ID = 198
NOM_GROUPE = "mon-binome"   # À vous : le nom de votre groupe

`client.submit` crée la soumission, envoie `agent.py` et la déploie.
La plateforme joue d'abord un match de vérification ; si l'agent
plante, dépasse le temps ou joue un coup interdit, la soumission
échoue et `last_status_message` dit pourquoi.

**Exécutez cette cellule une seule fois** : chaque exécution crée une
nouvelle soumission.

In [ ]:
if CHALLENGE_ID and os.environ.get("MLARENA_API_KEY"):
    import mlarena

    client = mlarena.connect(api_key=os.environ["MLARENA_API_KEY"])
    resultat = client.submit(challenge_id=CHALLENGE_ID, files=["agent.py"],
                             submission_name=f"puissance4-{NOM_GROUPE}")
    print(resultat)
else:
    print("Pas de clé MLARENA_API_KEY : soumission non envoyée.")

Le statut passe par `deploy_queue`, `deploy_run`, puis `active`. Une
fois active, la soumission joue des matchs contre celles des autres
groupes et sa note ELO évolue.

In [ ]:
if CHALLENGE_ID and os.environ.get("MLARENA_API_KEY"):
    statut = client.status()
    print(statut["status"], "|", statut["last_status_message"])
    print(client.leaderboard(CHALLENGE_ID).head(10))

---

## 8. Pour aller plus loin (facultatif) : Monte Carlo Tree Search

MCTS n'a pas besoin de fonction d'évaluation : il joue des parties
**au hasard** depuis les positions candidates et garde les coups qui
gagnent le plus souvent. Chaque itération a quatre étapes :
**sélection** (descendre l'arbre en choisissant l'enfant de meilleur
score UCB1), **expansion** (ajouter un coup pas encore essayé),
**simulation** (finir la partie au hasard), **rétropropagation**
(mettre à jour victoires et visites jusqu'à la racine).

Le score UCB1 d'un enfant équilibre **exploitation** (son taux de
victoire) et **exploration** (les enfants peu visités) :

$$\mathrm{UCB1} = \frac{g}{n} + c\sqrt{\frac{\ln N}{n}}$$

avec $g$ les victoires et $n$ les visites de l'enfant, $N$ les visites
du parent, et $c$ la constante d'exploration (souvent $\sqrt{2}$).

### Exercice 8 — `ucb1`

Renvoyez `float("inf")` si l'enfant n'a jamais été visité (il faut
l'essayer), sinon la formule.

In [ ]:
import math


def ucb1(gains, visites, visites_parent, c=1.41):
    """Le score UCB1 d'un enfant de l'arbre MCTS."""
    if visites == 0:
        return float("inf")
    return gains / visites + c * math.sqrt(math.log(visites_parent) / visites)

In [ ]:
assert ucb1(0, 0, 10) == float("inf")
assert abs(ucb1(3, 4, 10) - (0.75 + 1.41 * math.sqrt(math.log(10) / 4))) < 1e-9
assert ucb1(3, 4, 10, c=0) == 0.75, "c = 0 : exploitation pure"
assert ucb1(1, 2, 100) > ucb1(1, 2, 10), "plus le parent est visité, plus on explore"
print("✅ Exercice 8 réussi")

Un MCTS complet, construit sur `ucb1` (donné). Chaque nœud retient la
grille, le joueur qui vient de jouer, ses enfants, ses visites et ses
victoires (du point de vue de ce joueur).

In [ ]:
def autre(joueur):
    return ADV if joueur == MOI else MOI


class Noeud:
    def __init__(self, grille, joueur, parent=None, coup=None):
        self.grille, self.joueur = grille, joueur  # joueur : celui qui a joué coup
        self.parent, self.coup = parent, coup
        self.enfants, self.visites, self.gains = [], 0, 0.0
        fini = est_gagnant(grille, joueur) or not coups_valides(grille)
        self.a_essayer = [] if fini else coups_valides(grille)


def partie_au_hasard(grille, dernier):
    """Finit la partie au hasard ; renvoie le gagnant ou None (nulle)."""
    if est_gagnant(grille, dernier):
        return dernier
    g, joueur = grille.copy(), dernier
    while coups_valides(g):
        joueur = autre(joueur)
        col = random.choice(coups_valides(g))
        g[ligne_libre(g, col), col] = joueur
        if est_gagnant(g, joueur):
            return joueur
    return None


def mcts(grille, temps_max, c=1.41):
    racine = Noeud(grille, ADV)       # ADV vient de jouer : c'est à moi
    fin = time.perf_counter() + temps_max
    while time.perf_counter() < fin:
        noeud = racine
        while not noeud.a_essayer and noeud.enfants:        # 1. sélection
            parent = noeud
            noeud = max(parent.enfants,
                        key=lambda e: ucb1(e.gains, e.visites, parent.visites, c))
        if noeud.a_essayer:                                 # 2. expansion
            col = noeud.a_essayer.pop(random.randrange(len(noeud.a_essayer)))
            joueur = autre(noeud.joueur)
            enfant = Noeud(simuler_coup(noeud.grille, col, joueur), joueur,
                           noeud, col)
            noeud.enfants.append(enfant)
            noeud = enfant
        gagnant = partie_au_hasard(noeud.grille, noeud.joueur)  # 3. simulation
        while noeud is not None:                            # 4. rétropropagation
            noeud.visites += 1
            noeud.gains += 1.0 if gagnant == noeud.joueur else (
                0.5 if gagnant is None else 0.0)
            noeud = noeud.parent
    return max(racine.enfants, key=lambda e: e.visites).coup


class AgentMCTS(AgentAleatoire):
    def __init__(self, temps_max=0.08):
        self.temps_max = temps_max

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        return mcts(vers_grille(observation), self.temps_max)


taux = tournoi({"aléatoire": AgentAleatoire(), "règles": AgentRegles(),
                "MCTS": AgentMCTS(0.08), "minimax": AgentMinimax(3)}, 6)
print(taux.round(2))

| | Minimax | MCTS |
|---|---|---|
| évaluation | une heuristique à écrire | des parties au hasard |
| temps | profondeur fixe (ou itérative) | s'arrête n'importe quand |
| force | optimal si l'évaluation est parfaite | s'améliore avec le nombre de simulations |
| branchement élevé | souffre ($b^d$) | supporte bien |

En Python pur, MCTS ne fait que quelques centaines de simulations en
0,08 s : c'est peu, et il reste souvent derrière un minimax bien
élagué. Les programmes les plus forts combinent les deux idées :
vérifier les coups gagnants et les blocages immédiats, puis chercher
(AlphaGo associe MCTS et réseaux de neurones). Avec des *bitboards*,
les deux méthodes deviennent des dizaines de fois plus rapides.

---

**Bilan.** Votre agent anticipe, tient la limite de temps et joue sur
ML-Arena. Pour le projet : mettez ce code dans votre dépôt GitHub,
découpé en modules, avec ses tests `pytest`, un `pylint` propre et un
`README.md` qui explique vos choix.